# Evaluating Multilingual Factual Reliability of LLMs in Scientific Literature Analysis

**Author:** Aman Kumar (`mcl2026012@iiita.ac.in`)  
**Institution:** Department of Information Technology, Indian Institute of Information Technology, Allahabad  
**Course:** AI Tools for Research (AITR Lab 02)  
**Benchmark Anchor:** SciFact (Wadden et al., EMNLP 2020)  
**Evaluated Model:** `Qwen/Qwen2.5-0.5B-Instruct` (Deterministic greedy decoding, CPU)  

---

## Overview
This verified notebook reproduces the empirical findings, quantitative tables, and publication visualizations reported in the research manuscript. It demonstrates:
1. Schema verification and loading of genuine experimental outputs (`results/predictions.csv`).
2. Performance comparison between Direct Classification (Mode A) and Evidence-Grounded Classification (Mode B).
3. Matched-pair cross-lingual consistency and flip rate calculations across English, Spanish, and Hindi.
4. Confidence calibration analysis (ECE, Brier Score).
5. Reproduction of the 5 publication figures.

In [ ]:
import sys, os
# Ensure project root is in sys.path
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from src.metrics import evaluate_veracity_predictions, compute_cross_lingual_flip_rate, normalize_eval_label
from src.evaluation import verify_predictions_schema, compute_calibration_metrics

print("Evaluation modules and dependencies successfully imported.")

## 1. Load and Verify Genuine Experimental Predictions
We load `results/predictions.csv` containing the 24 evaluated passes ($4\text{ instances} \times 3\text{ languages} \times 2\text{ modes}$).

In [ ]:
preds_path = os.path.join('..', 'results', 'predictions.csv')
df_preds = pd.read_csv(preds_path)

is_valid, msg = verify_predictions_schema(df_preds)
print(f"Schema Valid: {is_valid} ({msg})")
print(f"Total Evaluated Records: {len(df_preds)}")

# Display first 6 records
df_preds[['claim_id', 'doc_id', 'language', 'prompt_mode', 'gold_veracity', 'pred_veracity', 'confidence', 'latency_seconds']].head(6)

## 2. Performance Breakdown by Prompting Mode
Comparing Direct Classification (Mode A) vs. Evidence-Grounded Classification (Mode B).

In [ ]:
mode_metrics_path = os.path.join('..', 'results', 'metrics_by_prompt.csv')
df_mode = pd.read_csv(mode_metrics_path)
df_mode

## 3. Matched-Pair Disagreement Analysis
Cross-lingual flip rate (CFR) across paired languages (`en` vs `es`, `en` vs `hi`, `es` vs `hi`) and cross-prompt mode disagreement (Mode A vs Mode B).

In [ ]:
disagree_path = os.path.join('..', 'results', 'paired_disagreement.csv')
df_disagree = pd.read_csv(disagree_path)
df_disagree

## 4. Visual Inspection of Publication Figures
The 5 publication-quality figures generated by `scripts/evaluate_predictions.py`.

In [ ]:
from IPython.display import Image, display

figures = [
    ('Figure 1: Accuracy & Macro-F1 across Languages and Prompt Modes', '../results/figures/fig1_accuracy_f1_by_condition.png'),
    ('Figure 2: Empirical Confusion Matrices (Mode A vs Mode B)', '../results/figures/fig2_confusion_matrices.png'),
    ('Figure 3: Runtime Latency Distributions', '../results/figures/fig3_runtime_by_condition.png'),
    ('Figure 4: Paired Disagreement Rates', '../results/figures/fig4_paired_disagreement.png'),
    ('Figure 5: Confidence Calibration Diagram & Reliability Curve', '../results/figures/fig5_calibration_analysis.png'),
]

for title, path in figures:
    print(f"\n=== {title} ===")
    if os.path.exists(path):
        display(Image(filename=path))
    else:
        print(f"Figure not found at {path}")